In [2]:
import pandas as pd


## 1. Borough-level disability aggregation, DAYS10P60GR and MONTHS_12



In [3]:
import numpy as np

def normalize_columns(df):
    rename_map = {}
    for col in df.columns:
        if col.lower() == "age16plus":
            rename_map[col] = "Age16plus"
    return df.rename(columns=rename_map)

def select_annual_weight(df, value_col, valid_codes):
    postal_mask = pd.to_numeric(df["mode"], errors="coerce").eq(2)
    postal_has_valid_data = df.loc[postal_mask, value_col].isin(valid_codes).any()
    return "wt_final" if postal_has_valid_data else "wt_final_online"

def weighted_binary_rate(sub, value_col, weight_col):
    valid = sub.dropna(subset=[value_col, weight_col])
    n = len(valid)
    if n == 0:
        return np.nan, 0, 0.0, 0.0
    w = valid[weight_col]
    weighted_n = w.sum()
    sum_w_squared = (w ** 2).sum()
    n_eff = (weighted_n ** 2) / sum_w_squared if sum_w_squared > 0 else 0.0
    if weighted_n == 0:
        return np.nan, n, 0.0, n_eff
    rate = ((valid[value_col] > 0).astype(int) * w).sum() / weighted_n
    return rate, n, weighted_n, n_eff

disab3_labels = {1: "limiting_disability", 2: "non_limiting_disability", 3: "no_disability"}

## 2. Aggregation function for a single year


In [4]:
def aggregate_borough_disability_days_months_year(path, year_number):
    year_df = pd.read_csv(path)
    year_df = normalize_columns(year_df)
    year_df = year_df[year_df["Age16plus"] == 1].copy()

    days_cols_y = [c for c in year_df.columns if c.startswith("DAYS10P60GR_")]
    months_cols_y = [c for c in year_df.columns if c.startswith("MONTHS_12_")]
    days_suffixes = set(c.replace("DAYS10P60GR_", "") for c in days_cols_y)
    months_suffixes = set(c.replace("MONTHS_12_", "") for c in months_cols_y)
    activities_y = sorted(days_suffixes & months_suffixes)
    disty_cols_y = [f"disty{i}_POP" for i in range(1, 14)]
    records = []

    for act in activities_y:
        days_weight = select_annual_weight(year_df, f"DAYS10P60GR_{act}", [0, 1, 2])
        months_weight = select_annual_weight(year_df, f"MONTHS_12_{act}", [0, 1])
        cols_needed = [f"DAYS10P60GR_{act}", f"MONTHS_12_{act}", "wt_final", "wt_final_online", "LA_2023"]
        sub_full = year_df[cols_needed + disty_cols_y + ["Disab3"]].copy()
        sub_full = sub_full.rename(columns={f"DAYS10P60GR_{act}": "DAYS10P60GR", f"MONTHS_12_{act}": "MONTHS_12"})

        for la_val, la_sub in sub_full.groupby("LA_2023"):
            for dv, dsub in la_sub[la_sub["Disab3"].isin([1, 2, 3])].groupby("Disab3"):
                r_days, n_days, wn_days, n_eff_days = weighted_binary_rate(dsub, "DAYS10P60GR", days_weight)
                r_months, n_months, wn_months, n_eff_months = weighted_binary_rate(dsub, "MONTHS_12", months_weight)
                records.append([year_number, la_val, disab3_labels[dv], act, r_days, n_days, wn_days, n_eff_days, r_months, n_months, wn_months, n_eff_months])

            for dcol in disty_cols_y:
                dsub = la_sub[la_sub[dcol] == 1]
                r_days, n_days, wn_days, n_eff_days = weighted_binary_rate(dsub, "DAYS10P60GR", days_weight)
                r_months, n_months, wn_months, n_eff_months = weighted_binary_rate(dsub, "MONTHS_12", months_weight)
                records.append([year_number, la_val, dcol.replace("_POP", ""), act, r_days, n_days, wn_days, n_eff_days, r_months, n_months, wn_months, n_eff_months])

    return pd.DataFrame(records, columns=["year", "LA_2023", "disability_group", "activity", "participation_DAYS10P60GR", "n_DAYS10P60GR", "weighted_n_DAYS10P60GR", "n_eff_DAYS10P60GR", "participation_MONTHS_12", "n_MONTHS_12", "weighted_n_MONTHS_12", "n_eff_MONTHS_12"])

## 3. Test on Year 7


In [5]:
path = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year7_125activities.csv"

year7_result = aggregate_borough_disability_days_months_year(path, 7)
print(year7_result.shape)
print(year7_result.head(20))


(64000, 12)
    year  LA_2023         disability_group       activity  \
0      7        8      limiting_disability  ABSEILING_H03   
1      7        8  non_limiting_disability  ABSEILING_H03   
2      7        8            no_disability  ABSEILING_H03   
3      7        8                   disty1  ABSEILING_H03   
4      7        8                   disty2  ABSEILING_H03   
5      7        8                   disty3  ABSEILING_H03   
6      7        8                   disty4  ABSEILING_H03   
7      7        8                   disty5  ABSEILING_H03   
8      7        8                   disty6  ABSEILING_H03   
9      7        8                   disty7  ABSEILING_H03   
10     7        8                   disty8  ABSEILING_H03   
11     7        8                   disty9  ABSEILING_H03   
12     7        8                  disty10  ABSEILING_H03   
13     7        8                  disty11  ABSEILING_H03   
14     7        8                  disty12  ABSEILING_H03   
15     7    

## 4. File paths for all eight years
Only the 125-activity version is used.


In [5]:
import os

root = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20"

year_files = {
    1: os.path.join(root, "Yifeng Mao", "data", "active_lives_1516_london_125.csv"),
    2: os.path.join(root, "Yifeng Mao", "data", "active_lives_1617_london_125.csv"),
    3: os.path.join(root, "Siyan Xin", "2017~2018", "2017_data_125_activities.csv"),
    4: os.path.join(root, "Siyan Xin", "2018~2019", "2018_data_125_activities.csv"),
    5: os.path.join(root, "Shuhan Zhao", "docs", "1920_london32_stable125.csv"),
    6: os.path.join(root, "Shuhan Zhao", "docs", "2021_london32_stable125.csv"),
    7: os.path.join(root, "Jingyi Hua", "data", "processed", "year7_125activities.csv"),
    8: os.path.join(root, "Jingyi Hua", "data", "processed", "year8_125activities.csv"),
}

for year_number, file_path in year_files.items():
    print(year_number, os.path.exists(file_path), file_path)


1 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Yifeng Mao\data\active_lives_1516_london_125.csv
2 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Yifeng Mao\data\active_lives_1617_london_125.csv
3 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Siyan Xin\2017~2018\2017_data_125_activities.csv
4 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Siyan Xin\2018~2019\2018_data_125_activities.csv
5 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\docs\1920_london32_stable125.csv
6 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\docs\2021_london32_stable125.csv
7 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year7_125activities.csv
8 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year8_125activities.csv


## 5. Check required columns before running all eight years


In [6]:
required_base_cols = ['Age16plus', 'Disab3', 'mode', 'wt_final', 'wt_final_online', 'LA_2023'] + [f'disty{i}_POP' for i in range(1, 14)]

def check_columns(path, year_number):
    cols = pd.read_csv(path, nrows=0).columns
    cols_lower = [c.lower() for c in cols]
    missing_base = [c for c in required_base_cols if c not in cols and c.lower() not in cols_lower]
    has_days = any(c.startswith('DAYS10P60GR_') for c in cols)
    has_months = any(c.startswith('MONTHS_12_') for c in cols)
    print(f"Year {year_number}: missing base columns = {missing_base}, has DAYS10P60GR = {has_days}, has MONTHS_12 = {has_months}")

for year_number, file_path in year_files.items():
    check_columns(file_path, year_number)


Year 1: missing base columns = [], has DAYS10P60GR = True, has MONTHS_12 = True
Year 2: missing base columns = [], has DAYS10P60GR = True, has MONTHS_12 = True
Year 3: missing base columns = [], has DAYS10P60GR = True, has MONTHS_12 = True
Year 4: missing base columns = [], has DAYS10P60GR = True, has MONTHS_12 = True
Year 5: missing base columns = [], has DAYS10P60GR = True, has MONTHS_12 = True
Year 6: missing base columns = [], has DAYS10P60GR = True, has MONTHS_12 = True
Year 7: missing base columns = [], has DAYS10P60GR = True, has MONTHS_12 = True
Year 8: missing base columns = [], has DAYS10P60GR = True, has MONTHS_12 = True


## 6. Run aggregation for all eight years and combine


In [7]:
all_years_results = []
problem_years = {}

for year_number in sorted(year_files.keys()):
    print('Processing year', year_number)
    try:
        year_result = aggregate_borough_disability_days_months_year(year_files[year_number], year_number)
        all_years_results.append(year_result)
    except KeyError as e:
        temp_df = pd.read_csv(year_files[year_number], nrows=5)
        print(f'Year {year_number} is missing column {e}')
        print('Columns containing la, disab, disty, wt, or age in this file:')
        print([col for col in temp_df.columns
               if 'la_' in col.lower() or 'disab' in col.lower() or 'disty' in col.lower()
               or 'wt' in col.lower() or 'age' in col.lower()])
        problem_years[year_number] = list(temp_df.columns)

if all_years_results:
    final_days_months_table = pd.concat(all_years_results, ignore_index=True)
    print(final_days_months_table.shape)

print('Years with problems:', list(problem_years.keys()))


Processing year 1
Processing year 2
Processing year 3
Processing year 4
Processing year 5
Processing year 6
Processing year 7
Processing year 8
(511488, 12)
Years with problems: []


## 7. Save the final table


In [8]:
output_dir = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed"
os.makedirs(output_dir, exist_ok=True)

final_output_path = os.path.join(output_dir, "RQ3_borough_disability_days_months_all_years.csv")
final_days_months_table.to_csv(final_output_path, index=False)
print('Saved to', final_output_path)


Saved to C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\RQ3_borough_disability_days_months_all_years.csv


In [6]:
import pandas as pd

WHITELIST_PATH = (
    r"C:\Users\Lenovo\Desktop\Dissertation\Data"
    r"\8_codebook\125_activities_composites_year1_to_year8.xlsx"
)

path_dm = (
    r"C:\Users\Lenovo\Documents\GitHub"
    r"\London-Sport2-Group20\Jingyi Hua\data\processed"
    r"\RQ3_borough_disability_days_months_all_years.csv"
)

path_mems = (
    r"C:\Users\Lenovo\Documents\GitHub"
    r"\London-Sport2-Group20\Jingyi Hua\data\processed"
    r"\RQ3_borough_disability_MEMS7GR_all_years.csv"
)

whitelist_df = pd.read_excel(
    WHITELIST_PATH,
    sheet_name="1_Stable composites",
    header=3
)

whitelist_125 = set(
    whitelist_df["DV suffix"]
    .dropna()
    .astype(str)
    .str.strip()
)

EXCLUDED_INCOMPLETE_ACTIVITIES = {"HULAHOOP_P27"}
common_124 = whitelist_125 - EXCLUDED_INCOMPLETE_ACTIVITIES

assert len(whitelist_125) == 125
assert "HULAHOOP_P27" in whitelist_125
assert len(common_124) == 124

df_dm = pd.read_csv(path_dm)

print("Rows before filtering:", len(df_dm))
print("Activities before filtering:", df_dm["activity"].nunique())

df_dm_clean = df_dm[
    df_dm["activity"].isin(common_124)
].copy()

print("Rows after filtering:", len(df_dm_clean))
print(
    "Activities after filtering:",
    df_dm_clean["activity"].nunique()
)

assert df_dm_clean["activity"].nunique() == 124
assert "HULAHOOP_P27" not in set(
    df_dm_clean["activity"].unique()
)
assert set(df_dm_clean["activity"].unique()) == common_124
assert len(df_dm_clean) == 507_904

df_mems_clean = pd.read_csv(path_mems)

assert df_mems_clean["activity"].nunique() == 124
assert set(df_mems_clean["activity"].unique()) == common_124
assert len(df_mems_clean) == 507_904

print(
    "Only in MEMS7GR:",
    set(df_mems_clean["activity"]) -
    set(df_dm_clean["activity"])
)

print(
    "Only in DAYS/MONTHS:",
    set(df_dm_clean["activity"]) -
    set(df_mems_clean["activity"])
)

df_dm_clean.to_csv(path_dm, index=False)

print("Saved to:", path_dm)

Rows before filtering: 507904
Activities before filtering: 124
Rows after filtering: 507904
Activities after filtering: 124
Only in MEMS7GR: set()
Only in DAYS/MONTHS: set()
Saved to: C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\RQ3_borough_disability_days_months_all_years.csv


In [7]:
for raw_col, neff_col in [
    ("n_DAYS10P60GR", "n_eff_DAYS10P60GR"),
    ("n_MONTHS_12", "n_eff_MONTHS_12"),
]:
    assert (df_dm_clean[neff_col] >= 0).all()
    assert (
        df_dm_clean[neff_col]
        <= df_dm_clean[raw_col] + 1e-8
    ).all()

assert df_dm_clean["activity"].nunique() == 124
assert len(df_dm_clean) == 507_904

print("All final checks passed.")

All final checks passed.
